In [50]:
# Cargamos las librerias
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


In [51]:
# ----------ESTACIONES----------
# abrimos el archivo de estaciones al que previamente hemos añadido las coordenadas en UTM
estaciones = pd.read_csv("../data/data_raw/M4_Estaciones_coordenadasUTM.csv",sep=";")
print(estaciones.columns)
estaciones.head(5)

Index(['OID_', 'OBJECTID', 'IDESTACION', 'FECHAACTUA', 'MODO', 'CODIGOESTA',
       'DENOMINACI', 'SITUACION', 'CODIGOCTME', 'CODIGOEMPR', 'DENOMINA_1',
       'MODOINTERC', 'CODIGOINTE', 'TIPO', 'CODIGOPROV', 'CODIGOMUNI',
       'CODIGOENTI', 'CODIGONUCL', 'CODIGOVIA', 'TIPOVIA', 'PARTICULA',
       'NOMBREVIA', 'TIPONUMERO', 'NUMEROPORT', 'CALIFICADO', 'CARRETERA',
       'CODIGOPOST', 'DISTRITO', 'SECCIONCEN', 'BARRIO', 'TESELA',
       'SECTORURBA', 'SECTOR', 'CORREDOR', 'CORONATARI', 'CORONA123',
       'ZONATRANSP', 'ENCUESTADO', 'ENCUESTAAF', 'HOJA25000', 'ACONDICION',
       'ACONDICI_1', 'FECHAALTA', 'FECHAINICI', 'FECHAFIN', 'X', 'Y',
       'GRADOACCES', 'SITUACIONC', 'DENOMINA_2', 'INTERURBAN', 'INTERURB_1',
       'LINEAS', 'POINT_Y', 'POINT_X'],
      dtype='object')


,OID_,OBJECTID,IDESTACION,FECHAACTUA,MODO,CODIGOESTA,DENOMINACI,SITUACION,CODIGOCTME,CODIGOEMPR,...,X,Y,GRADOACCES,SITUACIONC,DENOMINA_2,INTERURBAN,INTERURB_1,LINEAS,POINT_Y,POINT_X
0,0,1,4_153,20240628,4,153,ARROYOFRESNO,I,153,722,...,438581,4482705,T,,,,,7,"40,49089827","-3,7260354"
1,1,2,4_183,20240628,4,183,RIVAS URBANIZACIONES,S,183,924,...,453645,4468818,T,,,,,9,"40,3667715","-3,54727581"
2,2,3,4_38,20240628,4,38,NOVICIADO,I,50,211,...,440100,4475360,N,,,,,"2, 3, 10","40,42484223","-3,70741904"
3,3,4,4_199,20240628,4,199,LAGO,S,199,1013,...,437699,4474444,T,,,,,10,"40,41641411","-3,73563106"
4,4,5,4_200,20240628,4,200,BATAN,S,200,1014,...,436208,4473507,T,,,,,10,"40,40786021","-3,75310994"


In [52]:
# quitamos las columnas innecesarias
estaciones = estaciones.loc[:,['CODIGOESTA','DENOMINACI', 'CODIGOPOST', 'TIPOVIA', 
                               'PARTICULA', 'NOMBREVIA', 'TIPONUMERO', 'NUMEROPORT',
                               'CALIFICADO','CORONATARI', 'LINEAS', 'POINT_Y', 'POINT_X']]
# juntamos la direccion en una sola columna
estaciones["direccion"] = estaciones.TIPOVIA +" "+ estaciones.PARTICULA +" "+ estaciones.NOMBREVIA +" "+ estaciones.TIPONUMERO + estaciones.NUMEROPORT + estaciones.CALIFICADO
estaciones.drop(columns= ['TIPOVIA','PARTICULA', 'NOMBREVIA',
                          'TIPONUMERO', 'NUMEROPORT','CALIFICADO'],inplace=True)

In [53]:
# reordenamos y renombramos las columnas
estaciones = estaciones.loc[:,['CODIGOESTA', 'DENOMINACI', 'CODIGOPOST','direccion', 'CORONATARI', 'LINEAS',
       'POINT_Y', 'POINT_X']]
estaciones.columns = ['codigo', 'nombre', 'CP','direccion', 'zona', 'correspondencias',
       'latitud', 'longitud']

In [54]:
estaciones.head(5)

,codigo,nombre,CP,direccion,zona,correspondencias,latitud,longitud
0,153,ARROYOFRESNO,28035,Calle de Federica Montseny N2,A,7,"40,49089827","-3,7260354"
1,183,RIVAS URBANIZACIONES,28529,Plaza de Galicia N5,B1,9,"40,3667715","-3,54727581"
2,38,NOVICIADO,28015,Calle de San Bernardo N49,A,"2, 3, 10","40,42484223","-3,70741904"
3,199,LAGO,28011,Ronda del Lago N3,A,10,"40,41641411","-3,73563106"
4,200,BATAN,28011,Paseo de la Venta N1,A,10,"40,40786021","-3,75310994"


In [55]:
# vamos a poner el municipio al que pertenece cada estación
CP = pd.read_csv("../data/data_raw/listado-de-codigos-postales-de-españa.csv",sep=";",encoding="latin-1")
CP.columns = ["PROV", "poblacion","CP"]
estaciones = pd.merge(estaciones,CP, how="left", on="CP")
estaciones.columns


Index(['codigo', 'nombre', 'CP', 'direccion', 'zona', 'correspondencias',
       'latitud', 'longitud', 'PROV', 'poblacion'],
      dtype='object')

In [56]:
# reordenamos y guardamos 
estaciones = estaciones.loc[:,['codigo', 'nombre', 'CP', 'poblacion', 'direccion', 'zona', 'correspondencias',
       'latitud', 'longitud']]
estaciones.head(5)
estaciones.to_csv("../data/data_limpio/estaciones.csv")

In [57]:
# ----------DEMANDA DIARIA----------
# Abrimos el archivo descargado desde la CRTM 

ev_diaria = pd.read_excel("../data/data_raw/CRTM_Evolucion_demanda_diaria.xlsx", 
                          sheet_name="diaria", header=1)
ev_diaria.head(5)


,Unnamed: 0,Unnamed: 1,Metro de Madrid,EMT,Conc. por carretera,Renfe Cercanías,Total,Unnamed: 7,Nota: los datos de demanda reflejados son provisionales y como tal han de considerarse.
0,NaN,2023-01-01,685684,319488,155714,174991,1335877,NaN,NaN
1,NaN,2023-01-02,1581661,1024836,588003,446467,3640967,NaN,NaN
2,NaN,2023-01-03,1781186,1151845,662751,510268,4106050,NaN,NaN
3,NaN,2023-01-04,1846531,1160892,681347,517539,4206309,NaN,NaN
4,NaN,2023-01-05,1842966,1087828,615698,487856,4034348,NaN,NaN


In [58]:
# quitamos las columnas que no usamos
ev_diaria.drop(columns=["Unnamed: 0","Total","Unnamed: 7","Nota: los datos de demanda reflejados son provisionales y como tal han de considerarse."], inplace=True)
ev_diaria.columns

Index(['Unnamed: 1', 'Metro de Madrid', 'EMT', 'Conc. por carretera',
       'Renfe Cercanías'],
      dtype='object')

In [59]:
# renombramos columnas 
ev_diaria.columns = ['fecha', 'metro', 'EMT', 'conc_carretera',
       'cercanias']
ev_diaria.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1023 entries, 0 to 1022
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   fecha           1023 non-null   datetime64[ns]
 1   metro           1023 non-null   int64         
 2   EMT             1023 non-null   int64         
 3   conc_carretera  1023 non-null   int64         
 4   cercanias       1023 non-null   int64         
dtypes: datetime64[ns](1), int64(4)
memory usage: 40.1 KB


In [60]:
# separamos la fecha en dia mes y año
ev_diaria['dia'] = ev_diaria['fecha'].dt.day
ev_diaria['mes'] = ev_diaria['fecha'].dt.month
ev_diaria['año'] = ev_diaria['fecha'].dt.year

# sacamos el dia de la semana
ev_diaria['dia_semana'] = ev_diaria['fecha'].dt.weekday + 1

ev_diaria = ev_diaria.loc[:,['fecha', 'dia','mes', 'año', 'dia_semana', 
                             'metro', 'EMT', 'conc_carretera', 'cercanias']]

ev_diaria.head()

,fecha,dia,mes,año,dia_semana,metro,EMT,conc_carretera,cercanias
0,2023-01-01,1,1,2023,7,685684,319488,155714,174991
1,2023-01-02,2,1,2023,1,1581661,1024836,588003,446467
2,2023-01-03,3,1,2023,2,1781186,1151845,662751,510268
3,2023-01-04,4,1,2023,3,1846531,1160892,681347,517539
4,2023-01-05,5,1,2023,4,1842966,1087828,615698,487856


In [61]:
# veamos como se distribuyen los datos
ev_diaria.loc[:,["metro", "EMT", "conc_carretera", "cercanias"]].describe()

,metro,EMT,conc_carretera,cercanias
count,1.023000e+03,1.023000e+03,1.023000e+03,1023.000000
mean,1.910649e+06,1.304325e+06,7.919927e+05,518006.715543
std,5.076240e+05,4.302285e+05,2.934379e+05,151025.751760
min,6.694940e+05,3.194880e+05,1.557140e+05,174991.000000
25%,1.470529e+06,9.031760e+05,5.086940e+05,380551.500000
50%,2.059652e+06,1.468624e+06,8.803440e+05,568413.000000
75%,2.356417e+06,1.699372e+06,1.062361e+06,648772.000000
max,2.783341e+06,1.951608e+06,1.221809e+06,755825.000000


In [62]:
ev_diaria_dia_semana = pd.pivot_table(
    ev_diaria,
    values=['metro', 'EMT', 'conc_carretera', 'cercanias'],
    index='dia_semana',
    aggfunc='mean'
)
ev_diaria_dia_semana

ev_diaria_mes = pd.pivot_table(
    ev_diaria,
    values=['metro', 'EMT', 'conc_carretera', 'cercanias'],
    index='mes',
    aggfunc='mean'
)


In [63]:
# guardamos los dataset
ev_diaria.to_csv("../data/data_limpio/ev_diaria.csv")
ev_diaria_dia_semana.to_csv("../data/data_limpio/ev_diaria_dia_semana.csv")
ev_diaria_mes.to_csv("../data/data_limpio/ev_diaria_mes.csv")

In [64]:
# ----------ENTRADAS ESTACIONES----------
entradas = pd.read_excel("../data/data_raw/Ref._PA052_Entradas_estac_2025_09.xlsx", 
                          sheet_name="2025", header=0)
entradas.head(5)

,código_estación,nombre_estación,2025-01-01 00:00:00,2025-02-01 00:00:00,2025-03-01 00:00:00,2025-04-01 00:00:00,2025-05-01 00:00:00,2025-06-01 00:00:00,2025-07-01 00:00:00,2025-08-01 00:00:00,2025-09-01 00:00:00,2025-10-01 00:00:00,2025-11-01 00:00:00,2025-12-01 00:00:00
0,101,Plaza de Castilla,1307735,1331229,1412782,1252087,1325641,1317338,1206549,877266,1299144,NaN,NaN,NaN
1,102,Valdeacederas,254567,251967,275281,246902,263574,255278,227023,172968,254624,NaN,NaN,NaN
2,103,Tetuán,323903,321676,352715,320057,339147,331125,291227,222999,324467,NaN,NaN,NaN
3,104,Estrecho,450229,451408,493899,437368,471204,463197,407434,301194,457927,NaN,NaN,NaN
4,105,Alvarado,196198,191542,211058,188645,201768,204866,184625,148997,206295,NaN,NaN,NaN


In [ ]:
# lo primero que observamos es que el código de la estación no corresponde con el del archivo de estaciones. Este dódigo 
# parece tener más sentido porque los primeros dígitos coinciden # con los de las líneas de metro. Vamos a tomar este código
# como válido. Cambiamos el códidigo en el archivo de estaciones.

cod_estaciones = entradas[['código_estación', 'nombre_estación']].drop_duplicates()
cod_estaciones.columns = ['codigo', 'nombre']
cod_estaciones['nombre'] = cod_estaciones['nombre'].str.upper().str.replace('Á', 'A').str.replace('É', 'E').str.replace('Í', 'I').str.replace('Ó', 'O').str.replace('Ú', 'U')


estaciones = estaciones.drop(columns=['codigo'])
estaciones['nombre'] = estaciones['nombre'].str.replace('Á', 'A').str.replace('É', 'E').str.replace('Í', 'I').str.replace('Ó', 'O').str.replace('Ú', 'U')
estaciones = pd.merge(estaciones, cod_estaciones, how="left", on="nombre")
estaciones = estaciones.loc[:,['codigo', 'nombre', 'CP', 'poblacion','direccion', 'zona', 'correspondencias',
       'latitud', 'longitud']]



In [66]:
cod_estaciones.to_csv("../data/data_limpio/cod_estaciones.csv")

In [67]:
# veamos si hay estaciones sin código
estaciones[estaciones['codigo'].isnull()]

,codigo,nombre,CP,poblacion,direccion,zona,correspondencias,latitud,longitud
1,NaN,RIVAS URBANIZACIONES,28529,Rivas-Vaciamadrid,Plaza de Galicia N5,B1,9,"40,3667715","-3,54727581"
12,NaN,LA POVEDA,28500,Arganda del Rey,Parcela 9017 Polígono 1 SNS/N,B3,9,"40,31901573","-3,47744735"
28,NaN,ARGANDA DEL REY,28500,Arganda del Rey,Paseo de la Estación N37,B3,9,"40,30366928","-3,44751896"
63,NaN,RIVAS VACIAMADRID,28529,Rivas-Vaciamadrid,Calle Areneros N6,B2,9,"40,32837092","-3,52059752"
94,NaN,RIVAS FUTURA,28529,Rivas-Vaciamadrid,Calle Concepción Arenal N1,B2,9,"40,34134215","-3,52478901"
111,NaN,O'DONNELL,28028,Madrid,Calle del Doctor Esquerdo N47,A,6,"40,42288843","-3,66859515"
206,NaN,PARQUE LISBOA,28924,Alcorcón,Calle de Porto Lagos N7,B1,12,"40,34968728","-3,82119573"
266,NaN,VILLAVERDE BAJO CRUCE,28021,Madrid,Avda de Andalucía N38,A,3,"40,35089476","-3,69264894"


In [68]:
# hay 8 estaciones sin código. Vamos a asignarles el código correcto manualmente. Algunas estaciones no tienen códgigo porque no aparecen en el archivo de entradas, las dejaremos con 9999.
estaciones.loc[estaciones['nombre']=="RIVAS URBANIZACIONES", 'codigo'] = 9999
estaciones.loc[estaciones['nombre']=="LA POVEDA", 'codigo'] = 9999
estaciones.loc[estaciones['nombre']=="ESTADIO METROPOLITANO", 'codigo'] = 751
estaciones.loc[estaciones['nombre']=="ARGANDA DEL REY", 'codigo'] = 9999    
estaciones.loc[estaciones['nombre']=="RIVAS VACIAMADRID", 'codigo'] = 9999  
estaciones.loc[estaciones['nombre']=="RIVAS FUTURA", 'codigo'] = 9999
estaciones.loc[estaciones['nombre']=="O'DONNELL", 'codigo'] = 613   
estaciones.loc[estaciones['nombre']=="PARQUE LISBOA", 'codigo'] = 1202   
estaciones.loc[estaciones['nombre']=="VILLAVERDE BAJO CRUCE", 'codigo'] = 355   

# pasamos la columna código a tipo entero
estaciones['codigo'] = estaciones['codigo'].astype(int)

# no debe quedar ninguna estación sin código
estaciones[estaciones['codigo'].isnull()]

,codigo,nombre,CP,poblacion,direccion,zona,correspondencias,latitud,longitud


In [82]:
# volvemos a guardar el archivo de estaciones con los códigos corregidos y seguimos adelante con la limpieza de datos
estaciones.to_csv("../data/data_limpio/estaciones.csv")

In [ ]:
# se detecta en el archivo de estaciones que algunas estaciones tienen dos entradas porque aparecen dos o 
# más salidas de la estación. 
# Las eliminamos dejando una sola entrada por estación desde excel para poder comprobar mejor los datos.

In [ ]:
# de vuelta al archivo de entradas, nos interesa juntar los datos de todos los años que incluiremos en el estudio [2015-2025] 
# en un solo archivo:

entradas_historico = pd.read_excel("../data/data_raw/Ref._PA052_Entradas_estac_2025_09.xlsx", 
                          sheet_name="2015", header=0)
entradas_historico = pd.merge(entradas_historico, estaciones[["codigo","zona"]], how="left", left_on="código_estación", right_on="codigo")
entradas_historico = entradas_historico.loc[:,["código_estación", "nombre_estación", "zona"] + list(entradas_historico.columns[2:-2])]
entradas_historico.drop_duplicates(inplace=True)
entradas_historico.reset_index(drop=True, inplace=True)
entradas_historico.head(5)

,código_estación,nombre_estación,zona,2015-01-01 00:00:00,2015-02-01 00:00:00,2015-03-01 00:00:00,2015-04-01 00:00:00,2015-05-01 00:00:00,2015-06-01 00:00:00,2015-07-01 00:00:00,2015-08-01 00:00:00,2015-09-01 00:00:00,2015-10-01 00:00:00,2015-11-01 00:00:00,2015-12-01 00:00:00
0,101,Plaza de Castilla,A,982844.0,997328.0,1074249.0,1028264.0,1035880.0,1001026.0,905925.0,607344.0,982711.0,1111951.0,1060111.0,1007934.0
1,102,Valdeacederas,A,200599.0,204825.0,219247.0,210486.0,214752.0,207002.0,188927.0,132405.0,202454.0,231117.0,223057.0,217814.0
2,103,Tetuán,A,246994.0,249082.0,265117.0,254467.0,265918.0,256255.0,235950.0,166286.0,248698.0,286048.0,276058.0,266728.0
3,104,Estrecho,A,345734.0,343588.0,366077.0,349846.0,362671.0,348447.0,315850.0,220400.0,339915.0,401433.0,388476.0,377630.0
4,105,Alvarado,A,144302.0,142742.0,152046.0,146291.0,153679.0,147430.0,136095.0,98371.0,143906.0,165968.0,160541.0,161672.0


In [72]:
años = ["2016","2017","2018","2019","2020","2021","2022","2023","2024","2025"]

for año in años:
    entradas = pd.read_excel("../data/data_raw/Ref._PA052_Entradas_estac_2025_09.xlsx", 
                          sheet_name= año, header=0)
    entradas.drop(columns=["nombre_estación"], inplace=True)
    entradas_historico = pd.merge(entradas_historico, entradas, on="código_estación", how="outer")

entradas_historico.head(5)


,código_estación,nombre_estación,zona,2015-01-01 00:00:00,2015-02-01 00:00:00,2015-03-01 00:00:00,2015-04-01 00:00:00,2015-05-01 00:00:00,2015-06-01 00:00:00,2015-07-01 00:00:00,...,2025-03-01 00:00:00,2025-04-01 00:00:00,2025-05-01 00:00:00,2025-06-01 00:00:00,2025-07-01 00:00:00,2025-08-01 00:00:00,2025-09-01 00:00:00,2025-10-01 00:00:00,2025-11-01 00:00:00,2025-12-01 00:00:00
0,101,Plaza de Castilla,A,982844.0,997328.0,1074249.0,1028264.0,1035880.0,1001026.0,905925.0,...,1412782,1252087,1325641,1317338,1206549,877266,1299144,NaN,NaN,NaN
1,102,Valdeacederas,A,200599.0,204825.0,219247.0,210486.0,214752.0,207002.0,188927.0,...,275281,246902,263574,255278,227023,172968,254624,NaN,NaN,NaN
2,103,Tetuán,A,246994.0,249082.0,265117.0,254467.0,265918.0,256255.0,235950.0,...,352715,320057,339147,331125,291227,222999,324467,NaN,NaN,NaN
3,104,Estrecho,A,345734.0,343588.0,366077.0,349846.0,362671.0,348447.0,315850.0,...,493899,437368,471204,463197,407434,301194,457927,NaN,NaN,NaN
4,105,Alvarado,A,144302.0,142742.0,152046.0,146291.0,153679.0,147430.0,136095.0,...,211058,188645,201768,204866,184625,148997,206295,NaN,NaN,NaN


In [73]:
#quitamos las columnas con NaN
entradas_historico.dropna(how="all", axis=1, inplace=True)
entradas_historico.head(5)

,código_estación,nombre_estación,zona,2015-01-01 00:00:00,2015-02-01 00:00:00,2015-03-01 00:00:00,2015-04-01 00:00:00,2015-05-01 00:00:00,2015-06-01 00:00:00,2015-07-01 00:00:00,...,2024-12-01 00:00:00,2025-01-01 00:00:00,2025-02-01 00:00:00,2025-03-01 00:00:00,2025-04-01 00:00:00,2025-05-01 00:00:00,2025-06-01 00:00:00,2025-07-01 00:00:00,2025-08-01 00:00:00,2025-09-01 00:00:00
0,101,Plaza de Castilla,A,982844.0,997328.0,1074249.0,1028264.0,1035880.0,1001026.0,905925.0,...,1278273.0,1307735,1331229,1412782,1252087,1325641,1317338,1206549,877266,1299144
1,102,Valdeacederas,A,200599.0,204825.0,219247.0,210486.0,214752.0,207002.0,188927.0,...,253025.0,254567,251967,275281,246902,263574,255278,227023,172968,254624
2,103,Tetuán,A,246994.0,249082.0,265117.0,254467.0,265918.0,256255.0,235950.0,...,325551.0,323903,321676,352715,320057,339147,331125,291227,222999,324467
3,104,Estrecho,A,345734.0,343588.0,366077.0,349846.0,362671.0,348447.0,315850.0,...,449884.0,450229,451408,493899,437368,471204,463197,407434,301194,457927
4,105,Alvarado,A,144302.0,142742.0,152046.0,146291.0,153679.0,147430.0,136095.0,...,203053.0,196198,191542,211058,188645,201768,204866,184625,148997,206295


In [74]:
# ajustamos el formato de las columnas de fechas
columnas_modificar = entradas_historico.columns[3:]
fechas_dt = pd.to_datetime(columnas_modificar, format="%Y-%m")
entradas_historico.columns = ['codigo','nombre','zona'] + list(fechas_dt.strftime('%Y-%m'))
entradas_historico.head(5)


,codigo,nombre,zona,2015-01,2015-02,2015-03,2015-04,2015-05,2015-06,2015-07,...,2024-12,2025-01,2025-02,2025-03,2025-04,2025-05,2025-06,2025-07,2025-08,2025-09
0,101,Plaza de Castilla,A,982844.0,997328.0,1074249.0,1028264.0,1035880.0,1001026.0,905925.0,...,1278273.0,1307735,1331229,1412782,1252087,1325641,1317338,1206549,877266,1299144
1,102,Valdeacederas,A,200599.0,204825.0,219247.0,210486.0,214752.0,207002.0,188927.0,...,253025.0,254567,251967,275281,246902,263574,255278,227023,172968,254624
2,103,Tetuán,A,246994.0,249082.0,265117.0,254467.0,265918.0,256255.0,235950.0,...,325551.0,323903,321676,352715,320057,339147,331125,291227,222999,324467
3,104,Estrecho,A,345734.0,343588.0,366077.0,349846.0,362671.0,348447.0,315850.0,...,449884.0,450229,451408,493899,437368,471204,463197,407434,301194,457927
4,105,Alvarado,A,144302.0,142742.0,152046.0,146291.0,153679.0,147430.0,136095.0,...,203053.0,196198,191542,211058,188645,201768,204866,184625,148997,206295


In [75]:
entradas_historico.to_csv("../data/data_limpio/entradas_historico.csv")



In [76]:
fechas = [col for col in entradas_historico.columns if col >= '2023-01']
entradas_estacion = entradas_historico[fechas]
entradas_estacion.head()


,codigo,nombre,zona,2023-01,2023-02,2023-03,2023-04,2023-05,2023-06,2023-07,...,2024-12,2025-01,2025-02,2025-03,2025-04,2025-05,2025-06,2025-07,2025-08,2025-09
0,101,Plaza de Castilla,A,1129554.0,1216394.0,1387303.0,1203115.0,1306553.0,1308380.0,1102301.0,...,1278273.0,1307735,1331229,1412782,1252087,1325641,1317338,1206549,877266,1299144
1,102,Valdeacederas,A,223122.0,223395.0,245999.0,220152.0,243853.0,239021.0,195130.0,...,253025.0,254567,251967,275281,246902,263574,255278,227023,172968,254624
2,103,Tetuán,A,290262.0,294216.0,328687.0,294211.0,320425.0,314052.0,262444.0,...,325551.0,323903,321676,352715,320057,339147,331125,291227,222999,324467
3,104,Estrecho,A,412547.0,417930.0,464020.0,410553.0,449193.0,436191.0,343407.0,...,449884.0,450229,451408,493899,437368,471204,463197,407434,301194,457927
4,105,Alvarado,A,176883.0,176798.0,198171.0,180046.0,196250.0,189631.0,148897.0,...,203053.0,196198,191542,211058,188645,201768,204866,184625,148997,206295


In [77]:
# veamos si hay valores NaN o 0
nulos = entradas_estacion.isna().sum().sum()
cero = (entradas_estacion==0).sum().sum()
print(f"Nulos: {nulos}, Ceros: {cero}")

Nulos: 87, Ceros: 130


In [78]:
# Los valores 0 no son realistas, suelen responder a cierres temporales de estaciones por obras o mantenimiento Vamos a convertirlos en NaN para taparlos junto con los NaN
entradas_estacion = entradas_estacion.replace(0, pd.NA)

# Comprobamos que todo está correcto
nulos = entradas_estacion.isna().sum().sum()
cero = (entradas_estacion==0).sum().sum()
print(f"Nulos: {nulos}, Ceros: {cero}")


Nulos: 217, Ceros: 0


In [79]:
# Vamos a tapar los valores NaN con la media de entradas de esa estación  

# Calcular la media de cada columna
medias_estacion = [col for col in entradas_estacion.columns if col[:4].isdigit()]

# Rellenar los NaN con la media correspondiente 
entradas_estacion[medias_estacion] = entradas_estacion[medias_estacion].fillna(
    entradas_estacion[medias_estacion].mean()
)

# Comprobamos que todo está correcto
nulos = entradas_estacion.isna().sum().sum()
cero = (entradas_estacion==0).sum().sum()
print(f"Nulos: {nulos}, Ceros: {cero}")

Nulos: 0, Ceros: 0


C:\Users\user\AppData\Local\Temp\ipykernel_9200\2979959817.py:7: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  entradas_estacion[medias_estacion] = entradas_estacion[medias_estacion].fillna(


In [80]:
# Para nuestro estudio nos interesa tener la media mensual de entradas por estación. Vamos a calcularla y guardarla en un nuevo archivo.
entradas_estacion['media_miles'] = round(entradas_estacion[medias_estacion].mean(axis=1)/1000,2)
media_entradas = entradas_estacion.loc[:,['codigo','nombre','zona','media_miles']]
media_entradas.head()

,codigo,nombre,zona,media_miles
0,101,Plaza de Castilla,A,1273.37
1,102,Valdeacederas,A,237.58
2,103,Tetuán,A,308.42
3,104,Estrecho,A,427.20
4,105,Alvarado,A,185.87


In [81]:
# Guardamos el archivo limpio
media_entradas.to_csv("../data/data_limpio/media_entradas.csv")